# CUDA Kernel 面试主线 · 第 8/12 课：朴素转置与合并访存

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：正确索引非方阵转置，并从 warp 地址序列解释跨步写。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：矩阵转置交换行列；row-major 输入连续读后，直接写转置位置通常形成大步长 store。

## 核心心智模型

### 1. 它是什么，解决什么问题

矩阵转置交换行列；row-major 输入连续读后，直接写转置位置通常形成大步长 store。

### 2. 它如何工作

线程 (row,col) 读取 input[row*N+col]，写 output[col*M+row]；输出 leading dimension 是 M。

### 3. 正确性条件与常见误区

必须用 M、N 各自做边界和索引；只测方阵会掩盖 leading-dimension 错误。

### 4. 性能与工程取舍

朴素版代码简单；shared-memory tiled 版多同步却能让读写两端都合并。

## 具体演示

M=2,N=3 时 input[1,2] 线性索引 5，输出坐标 [2,1] 的线性索引 2*2+1=5。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐输出线性索引。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/08_transpose_naive.cu
#include <cuda_runtime.h>

// Naive Transpose:
// input:  [M, N] row-major
// output: [N, M] row-major
//
// 这个版本一个线程搬一个元素。
// 读 input 时，相邻线程读 input[row, col] 的连续地址，读是合并的；
// 写 output[col, row] 时，相邻线程写的地址跨度是 M，写通常不合并。
//
// 面试讲法：
// naive transpose 的瓶颈通常在 strided global store，
// 优化版使用 shared memory tile，把 strided store 变成 coalesced store。
__global__ void transpose_naive_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int M,
    int N
) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < M && col < N) {
        output[______] = input[row * N + col];  // TODO: 非方阵也正确
    }
}

void launch_transpose_naive(
    const float* input,
    float* output,
    int M,
    int N,
    cudaStream_t stream
) {
    dim3 block(16, 16);
    dim3 grid((N + block.x - 1) / block.x, (M + block.y - 1) / block.y);
    transpose_naive_kernel<<<grid, block, 0, stream>>>(input, output, M, N);
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/08_transpose_naive.cu -o /tmp/08_transpose_naive.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“朴素转置与合并访存”的工作机制。

**你的答案：**


### Q2

为何只用 1024×1024 方阵测试容易漏掉索引 bug？

**你的答案：**


### Q3

列主序输入时这套索引和合并访存结论如何变化？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。